# Bolt: GPU training on Google Colab (free T4)

A second free GPU when Kaggle's weekly 30 hours run out. **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.
Pick the run in the next cell. Everything is simulation; results land in `/content/out` and (if you allow it) your Google Drive,
in a folder `bolt-runs`. Nothing here calls an external AI service. The dog motion clips are CC BY-NC (non-commercial).

Free Colab sessions stop after a few hours or when the tab is idle: the runs below are 40 minutes (pilots) or ~3 h, and
every evaluation saves a checkpoint, so a cut-off run still leaves its latest brain.

In [ ]:
RUN = "privhist"   # one of: privhist, base798, sitpilot, jumppilot, lite3gpu1
HOURS = 0.7          # 0.7 = a 40-minute pilot; 3.0 for a full run (Colab may stop a session after a few hours)
SAVE_TO_DRIVE = True

In [ ]:
import subprocess, sys, os
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: Runtime -> Change runtime type -> T4 GPU")
if not os.path.isdir("/content/Cool-Websites"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "claude/robot-build-memory-ju76nt",
                    "https://github.com/MrArun005/Cool-Websites", "/content/Cool-Websites"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.9.2", "jaxlib==0.9.2", "flax==0.12.6", "optax==0.2.8",
                "orbax-checkpoint==0.12.8", "mujoco==3.17.0", "mujoco-mjx==3.17.0", "brax==0.14.2", "gymnasium==1.4.0"], check=True)
import jax; print("devices:", jax.devices())

In [ ]:
COMMON = ("--steps 2000000000 --envs 4096 --evals 10 --latency-min-ms 0 --latency-max-ms 40 --steer-max 0.8 "
          "--root-global 0.5 --steer-weight 0.6 --switch-rate 0.3 --moving-gaits walk trot --moving-clips sit lie "
          "--drop-p 0.05 --damping-range 0.1 2.0")
GO2 = COMMON + " --body go2 --blind-vel --reward-scaling 0.26880143601458134"
M = "../models"
RUNS = {   # the same runs as train/LOCAL_GPU_PROMPT.md and the Kaggle queue (deploy/PLAN.md)
    "base798":   f"{GO2} --moving-rate 0.5 --seed 61 --warm {M}/bolt-go2-damp-798M/params.pkl",
    "privhist":  f"{GO2} --moving-rate 0.5 --seed 61 --warm {M}/bolt-go2-damp-798M/params.pkl --privileged-critic --history 3",
    "sitpilot":  f"{GO2} --moving-rate 0.75 --seed 51 --warm {M}/bolt-go2-damp-798M/params.pkl",
    "jumppilot": f"{GO2} --moving-rate 0.5 --seed 43 --warm {M}/bolt-go2-jump-warm/params.pkl "
                 "--clips walk trot pace canter turn_left turn_right sit lie getup jump --clip-weight canter=3 jump=4",
    "lite3gpu1": f"{COMMON} --body lite3 --moving-rate 0.5 --motor-speed-range 1 4 --seed 71 --warm {M}/bolt-lite3-dog-61.8M/params.pkl",
}
cmd = f"python mjx_imitate.py train {RUNS[RUN]} --hours {HOURS} --out /content/out/{RUN}"
print(cmd)
os.chdir("/content/Cool-Websites/train")
p = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "update at" in line or "done" in line or "stopped" in line or "Error" in line or "body:" in line:
        print(line, end="", flush=True)
p.wait(); print("exit", p.returncode)

In [ ]:
# score on Colab's CPU (as every result in deploy/PLAN.md): the full exam, the moving sit/lie test, the jump
body = "lite3" if RUN.startswith("lite3") else "go2"
env = dict(os.environ, BOLT_BODY=body, OMP_NUM_THREADS="1", MUJOCO_GL="egl")
brain = f"/content/out/{RUN}/params.pkl"
for extra in ([], ["--moving"]) + ([["--jump"]] if RUN == "jumppilot" else []):
    r = subprocess.run([sys.executable, "go2_exam.py", brain, *extra], capture_output=True, text=True, env=env)
    print(*extra or ["full exam"], "->", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-400:])

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/bolt-runs/{RUN}"
    os.makedirs(dst, exist_ok=True)
    subprocess.run(["cp", "-r", f"/content/out/{RUN}/.", dst], check=True)
    print("saved to Google Drive:", dst, "(share the folder link with Claude, or download params.pkl and progress.json)")